# Active Model B — CNEEP_v2 Notebook

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
CNEEP_V2_ROOT = 'drive/MyDrive/CNEEP_v2/'

sys.path.append(CNEEP_V2_ROOT)
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data', 'AMB'))

from argparse import Namespace
import numpy as np
import torch
from datetime import datetime
from utils.sampler import CartesianSeqSampler
from utils.train import train
from utils.validate import validate
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from generate_trajectories import ActiveModelB

In [ ]:
#
# Hyper parameters
#
opt = Namespace()
opt.device = "cuda" if torch.cuda.is_available() else "cpu"

# alpha-NEEP
opt.alpha     = -0.5
opt.lam       = 0.0
opt.threshold = 0.01

opt.periodic    = True
opt.positional  = False
opt.latent_size = 10

# training
opt.n_iter           = 40
opt.train_batch_size = 1024
opt.test_batch_size  = 2048
opt.video_batch_size = 256
opt.n_hidden         = 512
opt.lr               = 1e-3
opt.wd               = 1e-5

opt.record_freq = 1000
opt.seed        = 3

# dataset / model architecture
opt.n_layer     = 4
opt.n_channel   = 32
opt.input_shape = (64, 64)    # AMB grid size
opt.M           = 1           # single trajectory
opt.L           = 48000
opt.L_test      = 48000
opt.seq_len     = 2
opt.time_step   = 0.001       # dt

# AMB model parameters (matching generate_animations.py defaults)
amb_params = dict(
    Lx=64, Ly=64, dx=1.0,
    a=0.25, b=0.25, kappa=4.0,
    lam=1.0, D=0.1, dt=0.001,
    smooth=False,
)
n_steps   = 48000
burn_in   = 50000
skip      = 1
init_mode = 'circle'

torch.manual_seed(opt.seed)

#
# results folder
#
result_folder = os.path.join(CNEEP_V2_ROOT, 'results')
current_result_folder = os.path.join(
    result_folder, f"AMB-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}")
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, 'model_parameter.pth.tar')

print(f"Device: {opt.device}")
print(f"Results: {current_result_folder}")

## 1. Generate AMB trajectory (train)

In [ ]:
train_seed = 42
np.random.seed(train_seed)

model_amb = ActiveModelB(**amb_params)

print(f"[INFO] Generating TRAIN trajectory (seed={train_seed}, burn_in={burn_in}, n_steps={n_steps})")
trajectory = model_amb.generate_trajectory(
    n_steps=n_steps, burn_in=burn_in, init_mode=init_mode
)
print(f"[INFO] Trajectory shape: {trajectory.shape}")

traj_train = trajectory   # skip=1, no subsampling
opt.L = traj_train.shape[0]
print(f"[INFO] Train data: {traj_train.shape}  (L={opt.L})")

## 1-b. Generate AMB trajectory (test, different seed)

In [ ]:
test_seed = 123
np.random.seed(test_seed)

model_amb_test = ActiveModelB(**amb_params)

print(f"[INFO] Generating TEST trajectory (seed={test_seed}, burn_in={burn_in}, n_steps={n_steps})")
trajectory_test = model_amb_test.generate_trajectory(
    n_steps=n_steps, burn_in=burn_in, init_mode=init_mode
)
print(f"[INFO] Test trajectory shape: {trajectory_test.shape}")

traj_test = trajectory_test   # skip=1, no subsampling
opt.L_test = traj_test.shape[0]
print(f"[INFO] Test data: {traj_test.shape}  (L_test={opt.L_test})")

In [ ]:
#
# Ground truth EPR (test data)
#
print("[INFO] Computing GT EPR time series on TEST data ...")
gt_total_epr = np.zeros(opt.L_test - 1)
gt_epr_maps  = np.zeros((opt.L_test - 1, amb_params["Lx"], amb_params["Ly"]))

traj_test_gpu = torch.tensor(traj_test).to(opt.device)
for t in tqdm(range(opt.L_test - 1)):
    epr_map = model_amb_test.compute_local_epr_density(traj_test_gpu[:, t], traj_test_gpu[:, t+1])
    gt_epr_maps[t] = epr_map
    gt_total_epr[t] = np.sum(epr_map) * model_amb_test.dx ** 2

gt_cumulative = np.cumsum(gt_total_epr * model_amb_test.dt)

print(f"GT mean EPR: {gt_total_epr.mean():.6e}")
print(f"GT cumulative EP (final): {gt_cumulative[-1]:.6e}")

## 2. Prepare video tensors (train & test)

In [ ]:
#
# Convert density field to video tensor: (M, L, 1, Lx, Ly)
#
train_video = torch.from_numpy(traj_train).float().to(opt.device)
train_video = train_video.unsqueeze(0).unsqueeze(2)   # (1, L, 1, Lx, Ly)

test_video = torch.from_numpy(traj_test).float().to(opt.device)
test_video = test_video.unsqueeze(0).unsqueeze(2)   # (1, L_test, 1, Lx, Ly)

mean = torch.mean(train_video)
std  = torch.std(train_video)
transform = lambda x: (x - mean) / std

print(f"Train video tensor: {train_video.shape}")
print(f"Test video tensor:  {test_video.shape}")
print(f"Mean: {mean:.4f}, Std: {std:.4f}")

## 3. Build and train model

In [ ]:
from models.CNEEP_0 import CNEEP

model = CNEEP(opt)
model = model.to(opt.device)
optim = torch.optim.Adam(model.parameters(), opt.lr, weight_decay=opt.wd)

train_sampler = CartesianSeqSampler(
    opt.M, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)
test_sampler  = CartesianSeqSampler(
    opt.M, opt.L_test, opt.seq_len, opt.test_batch_size, device=opt.device, train=False)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#
# Training loop
#
train_losses = []
R_values     = []
valid_losses = []
best_valid_loss = float('inf')

for i in tqdm(range(1, opt.n_iter + 1)):
    train_loss, R_value = train(
        opt, model, optim, train_video, train_sampler, transform)
    train_losses.append(train_loss)
    R_values.append(R_value)

    _, _, valid_loss = validate(
        opt, model, test_video, test_sampler, transform)
    valid_losses.append(valid_loss)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        state = {
            'epoch': i,
            'settings': opt.__dict__,
            'state_dict': model.state_dict(),
            'optimizer': optim.state_dict(),
        }
        torch.save(state, current_checkpoint_path)

model.load_state_dict(
    torch.load(current_checkpoint_path)['state_dict'])

In [ ]:
#
# Training curves
#
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(train_losses); axes[0].set_title('Train Loss')
axes[1].plot(valid_losses); axes[1].set_title('Valid Loss')
axes[2].plot(R_values);     axes[2].set_title('R (regularization)')
for ax in axes: ax.set_xlabel('Iteration')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/training_curves.png', dpi=150)
plt.show()

## 4. Validation — predicted EP

In [ ]:
#
# Full-trajectory validation (on TEST data)
#
full_sampler = CartesianSeqSampler(
    1, opt.L_test, opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)

pred_ent, pred_maps, _ = validate(
    opt, model, test_video, full_sampler, transform)

pred_maps = pred_maps / (amb_params['Lx'] * amb_params['Ly'])
pred_scalar = pred_ent.flatten()        # (L_test-1,) predicted EP per transition
pred_cumulative = np.cumsum(pred_scalar * amb_params['dt'])

print(f"Pred mean EP: {pred_scalar.mean():.6e}")
print(f"Pred cumulative EP (final): {pred_cumulative[-1]:.6e}")

## 5. EPR time series: GT vs Predicted

In [ ]:
#
# GT EPR vs Predicted EPR time series comparison
#
min_len = min(len(gt_total_epr), len(pred_scalar))
dt_eff  = amb_params['dt']
time_axis = np.arange(min_len) * dt_eff

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# (a) Instantaneous EPR
axes[0].plot(time_axis, gt_total_epr[:min_len],
             lw=0.3, alpha=0.6, color='steelblue', label='GT')
axes[0].plot(time_axis, pred_scalar[:min_len],
             lw=0.3, alpha=0.6, color='crimson', label='Predicted')
axes[0].set_ylabel(r'$\dot{S}_{\mathrm{tot}}(t)$', fontsize=12)
axes[0].set_title('Instantaneous EPR: GT vs Predicted', fontsize=13)
axes[0].legend(loc='upper right')
axes[0].axhline(0, color='k', lw=0.5, ls='--')

# (b) Cumulative EP
gt_cum  = np.cumsum(gt_total_epr[:min_len] * amb_params['dt'])
pred_cum = np.cumsum(pred_scalar[:min_len] * amb_params['dt'])
axes[1].plot(time_axis, gt_cum, lw=1.5, color='steelblue', label='GT')
axes[1].plot(time_axis, pred_cum, lw=1.5, color='crimson', label='Predicted')
axes[1].set_ylabel(r'$\sum \Delta S$', fontsize=12)
axes[1].set_title('Cumulative EP: GT vs Predicted', fontsize=13)
axes[1].legend(loc='upper left')

# (c) Running average EPR
window = max(min_len // 20, 1)
gt_running  = np.convolve(gt_total_epr[:min_len],
                          np.ones(window)/window, mode='valid')
pred_running = np.convolve(pred_scalar[:min_len],
                           np.ones(window)/window, mode='valid')
t_run = time_axis[:len(gt_running)]
axes[2].plot(t_run, gt_running, lw=1.2, color='steelblue', label='GT (running avg)')
axes[2].plot(t_run, pred_running, lw=1.2, color='crimson', label='Pred (running avg)')
axes[2].set_ylabel(r'$\langle \dot{S} \rangle$', fontsize=12)
axes[2].set_xlabel('Time', fontsize=12)
axes[2].set_title(f'Running average EPR (window={window})', fontsize=13)
axes[2].legend(loc='upper right')
axes[2].axhline(0, color='k', lw=0.5, ls='--')

fig.tight_layout()
fig.savefig(f'{current_result_folder}/epr_gt_vs_pred.png', dpi=150)
plt.show()

print(f"GT  mean EPR:   {gt_total_epr[:min_len].mean():.6e}")
print(f"Pred mean EPR:  {pred_scalar[:min_len].mean():.6e}")
print(f"Ratio (pred/GT): {pred_scalar[:min_len].mean() / (gt_total_epr[:min_len].mean() + 1e-20):.4f}")

In [ ]:
#
# Scatter plot: GT vs Predicted (per-transition)
#
from scipy import stats

gt_flat   = gt_total_epr[:min_len]
pred_flat = pred_scalar[:min_len]

slope, intercept, r_value, _, _ = stats.linregress(gt_flat, pred_flat)

plt.figure(figsize=(5, 5), dpi=100)
plt.scatter(gt_flat, pred_flat, alpha=0.05, s=3, color='grey', rasterized=True)
x_line = np.linspace(gt_flat.min(), gt_flat.max(), 100)
plt.plot(x_line, slope * x_line + intercept, 'r-', lw=2,
         label=f'$R^2 = {r_value**2:.4f}$')
plt.xlabel('GT total EPR')
plt.ylabel('Predicted EP')
plt.legend()
plt.title('Per-transition: GT vs Predicted')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/scatter_gt_vs_pred.png', dpi=150)
plt.show()
print(f'R² = {r_value**2:.6f}')

## 6. Mean EPR density map: GT vs Predicted

In [ ]:
#
# Mean local EPR density: GT vs Predicted
#
gt_mean_map   = gt_epr_maps[:min_len].mean(axis=0)
pred_mean_map = pred_maps[:min_len].mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

vmax = max(np.abs(gt_mean_map).max(), np.abs(pred_mean_map).max(), 1e-12)

im0 = axes[0].imshow(gt_mean_map, cmap='RdBu_r', origin='lower',
                       vmin=-vmax, vmax=vmax, interpolation='nearest')
axes[0].set_title('GT Mean Local EPR Density')
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(pred_mean_map, cmap='RdBu_r', origin='lower',
                       vmin=-vmax, vmax=vmax, interpolation='nearest')
axes[1].set_title('Predicted Mean Local EPR Density')
fig.colorbar(im1, ax=axes[1], shrink=0.8)

fig.suptitle('Time-averaged local EPR density', fontsize=13)
fig.tight_layout()
fig.savefig(f'{current_result_folder}/mean_epr_map_comparison.png', dpi=150)
plt.show()

## 7. EPR map animation

In [ ]:
#
# Overlay animation: density field + predicted EP map
#
n_frames = min(min_len, 400)

pred_map_norm = (pred_maps[:n_frames] - pred_maps[:n_frames].min()) / \
                (pred_maps[:n_frames].max() - pred_maps[:n_frames].min() + 1e-8)
density_np = traj_test[:n_frames]

fig, ax = plt.subplots(figsize=(5, 5))
im  = ax.imshow(density_np[0], cmap='gray', animated=True,
                origin='lower', interpolation='nearest')
ov  = ax.imshow(pred_map_norm[0], cmap='bwr', alpha=0.5,
                animated=True, origin='lower', interpolation='nearest')

def update(frame):
    im.set_array(density_np[frame])
    ov.set_array(pred_map_norm[frame])
    return [im, ov]

ani = FuncAnimation(fig, update, frames=n_frames, blit=True)
ani.save(f'{current_result_folder}/epr_overlay.mp4', fps=10)
plt.close()
print(f'Animation saved ({n_frames} frames)')

## 8. PCA of latent space

In [ ]:
latent_results = []
hooks = []

def hook_latent(module, input, output):
    latent_results.append(output.cpu().detach().numpy())

hooks.append(
    model._modules.get('latent')
    .register_forward_hook(hook_latent)
)

pca_sampler = CartesianSeqSampler(
    1, opt.L_test, opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)
_ = validate(opt, model, test_video, pca_sampler, transform)

for h in hooks:
    h.remove()

if latent_results:
    lv = latent_results[0] - np.mean(latent_results[0], axis=0)
    std_lv = np.std(lv, axis=0)
    std_lv[std_lv < 1e-8] = 1.0
    lv = lv / std_lv

    U, S, V = torch.pca_lowrank(torch.tensor(lv), q=opt.latent_size)

    plt.figure(figsize=(5, 5), dpi=100)
    colors = U[:, 2] if U.shape[1] > 2 else U[:, 0]
    colors = (colors - colors.mean()) / (colors.std() + 1e-8)
    plt.scatter(U[:, 0], U[:, 1], c=colors, cmap='viridis', s=5, alpha=0.5)
    plt.colorbar()
    plt.title('PCA of Latent Space')
    plt.xlabel('PC 1'); plt.ylabel('PC 2')
    plt.tight_layout()
    plt.savefig(f'{current_result_folder}/PCA_scatter.png', dpi=150)
    plt.show()

    np.set_printoptions(precision=2, suppress=True)
    print(f'Singular values: {S.numpy()}')
else:
    print('No latent vectors captured')